In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lower,
    to_date,
    to_timestamp,
    when,
    current_timestamp,
    from_json
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DecimalType,
    IntegerType
)

# ============================================
# STORAGE PATHS
# ============================================

bronze_root = (
    "abfss://bronze@bankingdelakevishal.dfs.core.windows.net/"
)

silver_root = (
    "abfss://silver@bankingdelakevishal.dfs.core.windows.net/"
)

In [0]:
# ============================================
# CONFIGURATION & SCHEMA DEFINITION
# ============================================
CATALOG_NAME = "banking_lakehouse_db2"          # Replace with your catalog name if different
SCHEMA_NAME = "silver"         # Dedicated schema for Silver layer tables
TABLE_NAME = "account"         # Standard singular table naming convention
FULL_TABLE_NAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"

# Ensure Silver schema exists in metastore
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")

# ============================================
# READ BRONZE DATA
# ============================================
account_bronze_df = spark.read.parquet(f"{bronze_root}account/")

# ============================================
# TRANSFORM & CLEANSE (BRONZE → SILVER)
# ============================================
account_silver_df = (
    account_bronze_df
    .select(
        trim(col("account_id")).alias("account_id"),
        trim(col("customer_id")).alias("customer_id"),
        trim(col("branch_id")).alias("branch_id"),
        trim(col("account_number")).alias("account_number"),
        upper(trim(col("account_type"))).alias("account_type"),
        upper(trim(col("currency"))).alias("currency"),
        trim(col("balance")).cast(DecimalType(18, 2)).alias("balance"),
        upper(trim(col("account_status"))).alias("account_status"),
        to_date(trim(col("opened_date")), "yyyy-MM-dd").alias("opened_date")
    )
    .filter(col("account_id").isNotNull())
    .dropDuplicates(["account_id"])
    .withColumn("silver_ingestion_timestamp", current_timestamp())
)

# ============================================
# WRITE FRESH DATA (OVERWRITE DELTA PATH & TABLE)
# ============================================
# 1. Overwrite raw Delta files in ADLS Silver storage
(
    account_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{silver_root}account/")
)

# 2. Register/Overwrite as a managed Unity Catalog table
(
    account_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE_NAME)
)

# ============================================
# VERIFICATION & METRICS
# ============================================
record_count = account_silver_df.count()
print(f"Successfully processed and wrote {record_count} records to '{FULL_TABLE_NAME}' and ADLS path '{silver_root}account/'.")

Successfully processed and wrote 200000 records to 'banking_lakehouse_db2.silver.account' and ADLS path 'abfss://silver@bankingdelakevishal.dfs.core.windows.net/account/'.


In [0]:
silver_account = spark.read.format("delta").load(
    "abfss://silver@bankingdelakevishal.dfs.core.windows.net/account/"
)

print("Record Count:", silver_account.count())

silver_account.printSchema()

display(silver_account.limit(10))

Record Count: 200000
root
 |-- account_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- branch_id: string (nullable = true)
 |-- account_number: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- balance: decimal(18,2) (nullable = true)
 |-- account_status: string (nullable = true)
 |-- opened_date: date (nullable = true)
 |-- silver_ingestion_timestamp: timestamp (nullable = true)



account_id,customer_id,branch_id,account_number,account_type,currency,balance,account_status,opened_date,silver_ingestion_timestamp
ACC0000018,CUST034905,BR0211,563743095616,SAVINGS,INR,3603535.75,CLOSED,2013-04-20,2026-09-02T20:06:15.845Z
ACC0000041,CUST033779,BR0093,231294868242,SAVINGS,INR,326885.37,ACTIVE,2023-05-23,2026-09-02T20:06:15.845Z
ACC0000043,CUST095160,BR0290,162937522512,SAVINGS,INR,2260600.51,CLOSED,2017-08-10,2026-09-02T20:06:15.845Z
ACC0000091,CUST045383,BR0192,654335833644,SAVINGS,INR,2187524.24,CLOSED,2013-08-03,2026-09-02T20:06:15.845Z
ACC0000126,CUST083420,BR0136,968757723725,SAVINGS,INR,2397158.05,CLOSED,2024-11-16,2026-09-02T20:06:15.845Z
ACC0000162,CUST074447,BR0322,508624173226,SAVINGS,INR,3792614.44,ACTIVE,2019-06-19,2026-09-02T20:06:15.845Z
ACC0000168,CUST021280,BR0053,262553522990,CURRENT,INR,1413295.33,CLOSED,2026-03-04,2026-09-02T20:06:15.845Z
ACC0000208,CUST090510,BR0426,671376456799,SAVINGS,INR,1759924.05,CLOSED,2011-07-31,2026-09-02T20:06:15.845Z
ACC0000229,CUST037512,BR0424,986097292059,SAVINGS,INR,3322655.72,ACTIVE,2013-07-31,2026-09-02T20:06:15.845Z
ACC0000236,CUST081646,BR0380,444329156732,SAVINGS,INR,3267783.68,ACTIVE,2019-06-11,2026-09-02T20:06:15.845Z


In [0]:
%sql
select * from banking_lakehouse_db2.silver.account where account_status = 'CLOSED'

account_id,customer_id,branch_id,account_number,account_type,currency,balance,account_status,opened_date,silver_ingestion_timestamp
ACC0000003,CUST035784,BR0042,401321128058,SAVINGS,INR,4888661.33,CLOSED,2024-05-07,2026-09-02T20:50:41.898Z
ACC0000006,CUST057127,BR0392,718321464449,CURRENT,INR,3147098.63,CLOSED,2011-12-31,2026-09-02T20:50:41.898Z
ACC0000014,CUST021125,BR0222,881537008801,SAVINGS,INR,2608518.56,CLOSED,2024-04-09,2026-09-02T20:50:41.898Z
ACC0000015,CUST066298,BR0363,235367700991,SAVINGS,INR,4393620.46,CLOSED,2016-10-14,2026-09-02T20:50:41.898Z
ACC0000016,CUST057346,BR0055,973508055687,CURRENT,INR,911479.63,CLOSED,2013-03-11,2026-09-02T20:50:41.898Z
ACC0000017,CUST025042,BR0122,527866748699,CURRENT,INR,2949456.56,CLOSED,2018-02-04,2026-09-02T20:50:41.898Z
ACC0000018,CUST034905,BR0211,563743095616,SAVINGS,INR,3603535.75,CLOSED,2013-04-20,2026-09-02T20:50:41.898Z
ACC0000020,CUST010309,BR0420,427350030285,SAVINGS,INR,1812402.14,CLOSED,2012-03-03,2026-09-02T20:50:41.898Z
ACC0000021,CUST037980,BR0412,991272082658,CURRENT,INR,3049148.50,CLOSED,2013-06-03,2026-09-02T20:50:41.898Z
ACC0000024,CUST013394,BR0010,362006761571,SAVINGS,INR,2997248.60,CLOSED,2019-09-13,2026-09-02T20:50:41.898Z
